In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Imports and Data loading 

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import(AutoTokenizer, AutoModelForSequenceClassification)
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print (f"Using device: {device}")

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
print(f"The shape of the train: {train.shape}")
print(f"The shape of the train: {test.shape}")
train_cc = train.copy()
test_cc = test.copy()
print(f"The shape of the copied train: {train_cc.shape}")
print(f"The shape of the copied test: {test_cc.shape}")

Using device: cuda
The shape of the train: (2000, 8)
The shape of the train: (500, 7)
The shape of the copied train: (2000, 8)
The shape of the copied test: (500, 7)


# Label Mapping

In [3]:
id_label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E",
}
label_id = {v: k for k, v in id_label.items()}

# Load Models

In [4]:
DEBERTA_MODEL = "microsoft/deberta-v3-small"
ROBERTA_MODEL = "roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MODEL,
    num_labels=5
).to(device)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_MODEL,
    num_labels=5
).to(device)

deberta_model.eval()
roberta_model.eval()

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [5]:
print(deberta_model.config.num_labels)
print(roberta_model.config.num_labels)

5
5


In [6]:
# Softmax helper
def get_probabilities(logits):
    return F.softmax(logits, dim=-1)

In [7]:
# MCQ formatting function
def build_inputs(row):
    prompt = str(row["prompt"])

    choices = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"]),
    ]

    return [prompt + " [SEP] " + choice for choice in choices]

In [8]:
# Top-3 prediction helper
def top3_predictions(probabilities):
    indices = np.argsort(probabilities)[::-1][:3]
    labels = [id_label[i] for i in indices]
    return " ".join(labels)

In [9]:
# Weighted Ensemble helper
def weighted_ensemble(
    deberta_probs,
    roberta_probs,
    w_deberta=0.7,
    w_roberta=0.3,
):
    return (
        w_deberta * deberta_probs
        + w_roberta * roberta_probs
    )

# Single model inference

In [10]:
@torch.no_grad()
def predict_probabilities(model, tokenizer, row, max_length=256):

    inputs = build_inputs(row)

    probs = []

    for text in inputs:

        encoding = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

        encoding = {k: v.to(device) for k, v in encoding.items()}

        logits = model(**encoding).logits

        probability = get_probabilities(logits).squeeze()

        if probability.numel() == 5:
            probs.append(probability.max().item())
        else:
            probs.append(probability[-1].item())

    probs = np.array(probs)
    probs = probs / probs.sum()

    return probs

# Questions and Answers

Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

# Question 1:

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?



(answer format : eg - A, probability of A)

In [11]:
row = train_cc.loc[25]
deberta_probs = predict_probabilities(
    deberta_model,
    deberta_tokenizer,
    row
)

print("DeBERTa Probabilities")
for i, p in enumerate(deberta_probs):
    print(f"{id_label[i]} : {p:.6f}")

best_idx = np.argmax(deberta_probs)

print("Top predictions:")
print(f"{id_label[best_idx]}, {deberta_probs[best_idx]:.6f}")

DeBERTa Probabilities
A : 0.200152
B : 0.199718
C : 0.200369
D : 0.199935
E : 0.199826
Top predictions:
C, 0.200369


Using the same sample (row index 25), average the class probabilities from both models.

`Average Probability = [P(DeBERTa) + P(RoBERTa)]/2`

# Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?

In [12]:
row = train_cc.loc[25]

roberta_probs = predict_probabilities(
    roberta_model,
    roberta_tokenizer,
    row
)

print("RoBERTa Probabilities")
for i, p in enumerate(roberta_probs):
    print(f"{id_label[i]} : {p:.6f}")

best_idx = np.argmax(roberta_probs)

print("\nTop Prediction:")
print(f"{id_label[best_idx]}, {roberta_probs[best_idx]:.6f}")

RoBERTa Probabilities
A : 0.200211
B : 0.199573
C : 0.199708
D : 0.200431
E : 0.200077

Top Prediction:
D, 0.200431


In [13]:
avg_probs = (deberta_probs + roberta_probs) / 2

print("Averaged Probabilities")
for i, p in enumerate(avg_probs):
    print(f"{id_label[i]} : {p:.6f}")

best_idx = np.argmax(avg_probs)

print("\nHighest Averaged Prediction:")
print(f"{id_label[best_idx]}, {avg_probs[best_idx]:.6f}")

Averaged Probabilities
A : 0.200181
B : 0.199645
C : 0.200038
D : 0.200183
E : 0.199952

Highest Averaged Prediction:
D, 0.200183


Apply weighted probability averaging after `Softmax` using the following weights:

`DeBERTa`: 0.70

`RoBERTa`: 0.30

Compute:

`P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]`

# Question 3:

Which answer option is ranked first after weighted `ensembling`?

In [14]:
def weighted_ensemble(
    deberta_probs,
    roberta_probs,
    w_deberta=0.7,
    w_roberta=0.3,
):
    return (
        w_deberta * deberta_probs
        + w_roberta * roberta_probs
    )

In [15]:
weighted_probs = weighted_ensemble(
    deberta_probs,
    roberta_probs,
    w_deberta=0.7,
    w_roberta=0.3
)

print("Weighted Ensemble Probabilities")

for i, p in enumerate(weighted_probs):
    print(f"{id_label[i]} : {p:.6f}")

best_idx = np.argmax(weighted_probs)

print("\nHighest Weighted Prediction:")
print(f"{id_label[best_idx]}, {weighted_probs[best_idx]:.6f}")

Weighted Ensemble Probabilities
A : 0.200169
B : 0.199674
C : 0.200170
D : 0.200084
E : 0.199902

Highest Weighted Prediction:
C, 0.200170


Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

# Question 4:

What is the Top-3 prediction string for row index 25?
Example -- C A E


In [16]:
print("Top-3 Prediction:")
print(top3_predictions(weighted_probs))

Top-3 Prediction:
C A D


Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

# Question 5:

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [17]:
predictions = []

for _, row in tqdm(test_cc.iterrows(), total=len(test_cc)):

    deberta_probs = predict_probabilities(
        deberta_model,
        deberta_tokenizer,
        row
    )

    roberta_probs = predict_probabilities(
        roberta_model,
        roberta_tokenizer,
        row
    )

    final_probs = weighted_ensemble(
        deberta_probs,
        roberta_probs,
        w_deberta=0.7,
        w_roberta=0.3
    )

    predictions.append(top3_predictions(final_probs))


submission = pd.DataFrame({
    "id": test_cc["id"],
    "prediction": predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"\nTotal predictions: {len(submission)}")

  0%|          | 0/500 [00:00<?, ?it/s]

   id prediction
0   1      B E A
1   2      E B D
2   3      A C D
3   4      D B E
4   5      C A D

Total predictions: 500


For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: _"Answer the following multiple-choice question carefully:"_

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

# Question 6:

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [18]:
def build_inputs_augmented(row):
    instruction = "Answer the following multiple-choice question carefully: "

    prompt = instruction + str(row["prompt"])

    choices = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"]),
    ]

    return [prompt + " [SEP] " + choice for choice in choices]

In [19]:
@torch.no_grad()
def predict_from_inputs(model, tokenizer, inputs, max_length=256):

    probs = []

    for text in inputs:

        encoding = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

        encoding = {k: v.to(device) for k, v in encoding.items()}

        logits = model(**encoding).logits

        probability = get_probabilities(logits).squeeze()

        if probability.numel() == 5:
            probs.append(probability.max().item())
        else:
            probs.append(probability[-1].item())

    probs = np.array(probs)
    probs /= probs.sum()

    return probs

In [20]:
changed = 0

for _, row in test_cc.head(50).iterrows():

    # Original prompt
    original_inputs = build_inputs(row)
    original_probs = predict_from_inputs(
        deberta_model,
        deberta_tokenizer,
        original_inputs
    )

    augmented_inputs = build_inputs_augmented(row)
    augmented_probs = predict_from_inputs(
        deberta_model,
        deberta_tokenizer,
        augmented_inputs
    )

    tta_probs = (original_probs + augmented_probs) / 2

    original_top1 = np.argmax(original_probs)
    tta_top1 = np.argmax(tta_probs)

    if original_top1 != tta_top1:
        changed += 1

print("Number of changed Top-1 predictions:", changed)

Number of changed Top-1 predictions: 20


Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

# Question 7:

How many rows have different Top-1 predictions?

In [21]:
different = 0

for _, row in test_cc.head(100).iterrows():

    deberta_probs = predict_probabilities(
        deberta_model,
        deberta_tokenizer,
        row
    )

    deberta_top1 = np.argmax(deberta_probs)

    roberta_probs = predict_probabilities(
        roberta_model,
        roberta_tokenizer,
        row
    )

    ensemble_probs = weighted_ensemble(
        deberta_probs,
        roberta_probs,
        w_deberta=0.7,
        w_roberta=0.3
    )

    ensemble_top1 = np.argmax(ensemble_probs)

    if deberta_top1 != ensemble_top1:
        different += 1

print("Different Top-1 predictions:", different)

Different Top-1 predictions: 6


For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

# Question 8:

How many rows have a positive confidence gain (greater than 0)?

In [22]:
positive_gain = 0

for _, row in test_cc.head(100).iterrows():

    deberta_probs = predict_probabilities(
        deberta_model,
        deberta_tokenizer,
        row
    )

    roberta_probs = predict_probabilities(
        roberta_model,
        roberta_tokenizer,
        row
    )

    ensemble_probs = weighted_ensemble(
        deberta_probs,
        roberta_probs,
        w_deberta=0.7,
        w_roberta=0.3
    )

    deberta_confidence = np.max(deberta_probs)
    ensemble_confidence = np.max(ensemble_probs)

    if ensemble_confidence > deberta_confidence:
        positive_gain += 1

print("Rows with positive confidence gain:", positive_gain)

Rows with positive confidence gain: 1


For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

# Question 9:

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

Examples -- A C D vs A D C

In [23]:
changed_top3 = 0

for _, row in test_cc.head(100).iterrows():

    deberta_probs = predict_probabilities(
        deberta_model,
        deberta_tokenizer,
        row
    )

    deberta_top3 = top3_predictions(deberta_probs)

    roberta_probs = predict_probabilities(
        roberta_model,
        roberta_tokenizer,
        row
    )

    ensemble_probs = weighted_ensemble(
        deberta_probs,
        roberta_probs,
        w_deberta=0.7,
        w_roberta=0.3
    )

    ensemble_top3 = top3_predictions(ensemble_probs)

    if deberta_top3 != ensemble_top3:
        changed_top3 += 1

print("Rows with changed Top-3 ranking:", changed_top3)

Rows with changed Top-3 ranking: 18


Using the Top-3 predictions generated by your weighted ensemble for the **first 100 validation samples**, compute the **MAP@3** score.

# Question 10:

What is the final **MAP@3 score**?
(Round to 4 decimal places)

In [24]:
def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):
        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score


scores = []

for _, row in train_cc.head(100).iterrows():

    deberta_probs = predict_probabilities(
        deberta_model,
        deberta_tokenizer,
        row
    )

    roberta_probs = predict_probabilities(
        roberta_model,
        roberta_tokenizer,
        row
    )

    ensemble_probs = weighted_ensemble(
        deberta_probs,
        roberta_probs,
        w_deberta=0.7,
        w_roberta=0.3
    )

    top3 = top3_predictions(ensemble_probs).split()

    scores.append(
        apk(row["answer"], top3, k=3)
    )

map3 = np.mean(scores)

print("MAP@3:", round(map3, 4))

MAP@3: 0.3717
